In [ ]:
run_count = 0

In [ ]:
import numpy as np
import pandas as pd
from itertools import product
import matplotlib.pyplot as plt
import seaborn as sns

Okay, this is where I got some help from chatgpt. The lattice class can now measure energies using arbitrary interactions, including long-range 1/r and 1/r^2 scaling

In [ ]:
import numpy as np
from collections import deque

class lattice:
    
    def __init__(self, num_dim, size):
        self._is_init = False
        
        # Initialization flag
        self._is_init = True
        # Number of dimensions
        self._num_dim = int(num_dim)
        # Lattice size
        self._size = np.array(size, dtype=int)
        # Total number of sites (volume)
        self._volume = 1
        for i in range(num_dim):
            self._volume *= size[i]
        self._volume = int(self._volume)
        
        # Forward and backward nearest neighbor index arrays (periodic boundary conditions)
        self._neighbor_forward = np.zeros((self._volume, self._num_dim), dtype=int)
        self._neighbor_backward = np.zeros((self._volume, self._num_dim), dtype=int)
        
        # Calculate nearest neighbors (1st order)
        self._neighbors = {}  # Dictionary to store neighbors of all orders
        self._neighbors[1] = [[] for _ in range(self._volume)]  # 1st order neighbors
        
        for i in range(self._volume):
            r = self.map_index_to_vec(i)
            nn_set = set()  # Use a set to avoid duplicates
            for j in range(self._num_dim):
                # Forward neighbor
                q_fwd = r.copy()
                q_fwd[j] = (q_fwd[j] + 1) % self._size[j]
                fwd_idx = self.map_vec_to_index(q_fwd)
                self._neighbor_forward[i, j] = fwd_idx
                nn_set.add(fwd_idx)
                
                # Backward neighbor
                q_bwd = r.copy()
                q_bwd[j] = (q_bwd[j] + self._size[j] - 1) % self._size[j]
                bwd_idx = self.map_vec_to_index(q_bwd)
                self._neighbor_backward[i, j] = bwd_idx
                nn_set.add(bwd_idx)
            self._neighbors[1][i] = list(nn_set)
        
        # Precompute neighbors up to a certain order (e.g., 2nd order)
        self._max_order = 2  # You can change this as needed
        self._compute_neighbors_up_to_order(self._max_order)
    
    def _compute_neighbors_up_to_order(self, max_order):
        """Compute neighbors up to max_order using BFS."""
        for current_order in range(2, max_order + 1):
            if current_order not in self._neighbors:
                self._neighbors[current_order] = [[] for _ in range(self._volume)]
            
            for i in range(self._volume):
                # Start BFS from site i
                visited = set()
                queue = deque()
                # Initialize with 1st order neighbors
                queue.append((i, 0))  # (site, distance)
                visited.add(i)
                neighbors_at_order = set()
                
                while queue:
                    current_site, distance = queue.popleft()
                    if distance == current_order - 1:
                        # The neighbors of current_site are at distance = current_order
                        for neighbor in self._neighbors[1][current_site]:
                            if neighbor not in visited:
                                neighbors_at_order.add(neighbor)
                    else:
                        for neighbor in self._neighbors[1][current_site]:
                            if neighbor not in visited:
                                visited.add(neighbor)
                                queue.append((neighbor, distance + 1))
                
                self._neighbors[current_order][i] = list(neighbors_at_order)
    
    def get_neighbors(self, site, order):
        """Return the neighbors of a given site at a specified order."""
        if order < 1:
            return []
        if order > self._max_order:
            self._compute_neighbors_up_to_order(order)
            self._max_order = order
        return self._neighbors[order][site]
    
    def map_index_to_vec(self, index):
        """Map a linear index to a position vector on the lattice."""
        vec = np.zeros(self._num_dim, dtype=int)
        for i in range(self._num_dim):
            vec[i] = index % self._size[i]
            index = index // self._size[i]
        return vec
    
    def map_vec_to_index(self, vec):
        """Map a position vector to a linear index on the lattice."""
        index = 0
        stride = 1
        for i in range(self._num_dim):
            index += vec[i] * stride
            stride *= self._size[i]
        return index
    
    def set_interactions(self, strengths):
        """Set the interaction strengths for each neighbor order.
        Args:
            strengths (dict): A dictionary where keys are neighbor orders (1, 2, ...)
                              and values are the corresponding coupling strengths (J1, J2, ...).
        Example:
            set_interactions({1: -1.0, 2: 0.5})  # J1 = -1.0, J2 = 0.5
        """
        self.couplings = strengths

    def compute_energy(self, isite, iq=None):
        """Compute the energy contribution of a site (or a hypothetical state).
        Args:
            isite (int): The index of the site.
            iq (optional): The hypothetical state to evaluate. If None, use the current state.
        Returns:
            float: The energy contribution of the site.
        """
        s_i = self._states[isite] if iq is None else iq

        # On-site energy term (e.g., external field)
        energy_i = self._A * s_i  # Assuming self._A is the on-site coefficient

        # Interaction terms for each neighbor order
        for order, J in self.couplings.items():
            interaction = 0.0
            neighbors = self.get_neighbors(isite, order)
            for j in neighbors:
                s_j = self._states[j]
                interaction += abs(s_i - s_j)  # Energy from difference with neighbors
            energy_i += (J / 2.0) * interaction  # Divide by 2 to avoid double-counting

        return energy_i
    
    def calculate_total_energy(self):
        total_energy = 0.0
        for i in range(self._volume):
            site_energy = self.compute_energy(i)
            total_energy += site_energy
        return total_energy
    
    def calculate_delta_energy(self, isite):
        '''for Monte-Carlo methods'''
        s_i = self._states[isite]
        s_prime_i = 1 - s_i 

        energy_initial = self.compute_energy(isite, iq=s_i)
        energy_final = self.compute_energy(isite, iq=s_prime_i)

        delta_e = energy_final - energy_initial

        return delta_e
    
    def make_hot_start(self, A = 1.0, fraction = 0.5 ):
        self._A = A
        states = (np.random.rand(self._volume) < fraction).astype(int)
        self._states = states
    
    def make_cold_start( self, A = 1.0, up=True ):
        self._A = A
        states = np.ones(self._volume, dtype=int)*up
        self._states = states



In [ ]:
def power_lrf(strength, const=2, cutoff='alpha', alpha=0.01):
    i = 1
    if cutoff == 'alpha':
        cutoff = -np.log(alpha)/np.log(const)
    
    couplings = {}
    while i < cutoff:
        couplings[i] = strength*(1/const)**i
        i += 1

    return couplings
        
def linear_lrf(strength, factor=None, cutoff='alpha', alpha=0.01):
    i = 1
    if cutoff == 'alpha':
        cutoff = 1/alpha
    if not factor:
        factor = 1

    couplings = {}
    while i < cutoff:
        couplings[i] = strength*(1/i**factor)
        i += 1

    return couplings


dim = 1
N = 16
chain = lattice(dim, [N])
chain.make_hot_start(A=0.03)
chain.set_interactions(linear_lrf(4, cutoff=4))

tau = 5.5
beta = 1/tau
boltzmann_factors = []
excited_states = []

energies = []
all_chain_states = [np.array([int(bit) for bit in f'{i:0{N}b}']) for i in range(2**N)]
for state in all_chain_states:
    chain._states = state
    E = chain.calculate_total_energy()
    energies.append(E)
    boltzmann_factors.append(np.exp(-E*beta))
    excited_states.append(sum(state))
Z = sum(boltzmann_factors)

chain_dict = {'state':all_chain_states, 'energy':energies, 'b_factors':boltzmann_factors, 'excited_states':excited_states}

All_States = pd.DataFrame(chain_dict)
All_States['probability'] = All_States['b_factors']/Z

number_excited = []
for i in range(N):
    state_probability = sum(All_States.loc[All_States['excited_states'] == i]['probability'])
    number_excited.append((i, state_probability))

prob_array = np.array(number_excited)
plt.axvline(N/2, color='red')
plt.plot(prob_array[:,0], prob_array[:, 1], label='probability')
plt.xlabel('excited states')
plt.legend()
plt.grid()

In [ ]:
# Production code

dim = 1
N = 12 # start small

A = 1

chain = lattice(dim, [N])
chain.make_cold_start(A=A)

meta_df = pd.DataFrame(columns=['state', 'b_factors', 'excited_states', 'probability', 'tau'])

tau_vals = np.linspace(1, 8, 30)

g = 4.0
const = 1.5
rcut = 20
interaction = power_lrf(g, const, cutoff=rcut)
chain.set_interactions(interaction)


for tau in tau_vals:
    beta = 1/tau

    boltzmann_factors = []
    excited_states = []
    all_chain_states = [np.array([int(bit) for bit in f'{i:0{N}b}']) for i in range(2**N)]
    
    for state in all_chain_states:
        chain._states = state
        E = chain.calculate_total_energy()
        boltzmann_factors.append(np.exp(-E*beta))
        excited_states.append(sum(state))

    Z = sum(boltzmann_factors)
    chain_dict = {'state':all_chain_states, 'b_factors':boltzmann_factors, 'excited_states':excited_states}
    All_States = pd.DataFrame(chain_dict)
    All_States['probability'] = All_States['b_factors']/Z
    All_States['tau'] = tau
    meta_df = pd.concat([meta_df, All_States], ignore_index=True, sort=False)

meta_df.head()

fig, ax = plt.subplots(1,1)


tau = tau_vals[0]
tau_states = meta_df[meta_df['tau'] == tau]
state_prob = []
for i in range(N):
    state_probability = sum(tau_states.loc[tau_states['excited_states'] == i]['probability'])
    state_prob.append(state_probability)

line, = plt.plot([i for i in range(N)], state_prob, label=fr'$\tau=$ {tau}')
ax.legend()
plt.xlabel('excited sites')
plt.ylabel('probability')
plt.title(fr'Interaction g=${g}/{const}^r$, rcut={rcut}')
#plt.title(fr'Interaction g=${g}/r$')

def update_line(ax, tau):
    tau_states = meta_df[meta_df['tau'] == tau]
    state_prob = []
    for i in range(N):
        state_probability = sum(tau_states.loc[tau_states['excited_states'] == i]['probability'])
        state_prob.append(state_probability)
    
    line.set_ydata(state_prob)
    line.set_label(fr'$\tau=$ {tau:.3f}')
    ax.legend()

ax.set_yscale('log')

In [ ]:
from animator import ParticleAnimator
from matplotlib.animation import FFMpegWriter # requires pillowwriter package
filename = f'{run_count}. {N}-chain brute-force, A={A},  t={min(tau_vals)}-{max(tau_vals)}.mp4'
run_count += 1

total_time = 5 # secondsl
fps = len(tau_vals)/total_time
print(fps)

anim = ParticleAnimator(fig, FFMpegWriter, filename)
anim.set_display(fps, total_time)
frame_data = anim.generate_frame_data(tau_vals)
anim.animate_ax(ax, update_line, frame_data)
anim.make_animation()
anim.display()

In [ ]:
def kinetic_monte_carlo(binary_lat, beta, total_time):
    t = 0.0
    time_series = [(t, binary_lat._states.copy())]
    
    while t < total_time:
        rates = []
        deltas = []
        indices = []
        
        # Compute rate for each possible flip
        for i in range(binary_lat._volume):
            delta_E = binary_lat.calculate_delta_energy(i)
            rate = np.exp(-beta * delta_E)
            rates.append(rate)
            deltas.append(delta_E)
            indices.append(i)
        
        R_total = sum(rates)
        if R_total == 0:
            break  # No moves possible

        # Sample time increment
        dt = -np.log(np.random.rand()) / R_total
        t += dt

        # Choose site to flip
        r = np.random.rand() * R_total
        acc = 0.0
        for i, rate in enumerate(rates):
            acc += rate
            if acc >= r:
                binary_lat._states[indices[i]] = 1 - binary_lat._states[indices[i]]
                break
        
        time_series.append((t, binary_lat._states.copy()))
    
    return time_series


tau = 4.2
beta = 1/tau
N = 100

chain = lattice(1, [N])
chain.set_interactions(interaction)
chain.make_cold_start(A=A, up=True)

dynamics = kinetic_monte_carlo(binary_lat=chain, beta=beta, total_time=60)
times = [a for a, b in dynamics]
states = [b for a,b in dynamics]

Dt = np.array(times)[1:]-np.array(times)[:-1]
sns.histplot(Dt)

Since Kinetic Monte Carlo has variable time steps, we use scipy linear interpolation to fill in data between timesteps.

In [ ]:
import numpy as np
from scipy.interpolate import interp1d

time = times[-1]
fps = 30
frames = int(time*fps)

t_regular = np.linspace(0, time, frames)

# Convert list of states to array
y_array = np.array(states)  # shape: (n_times, ...) 

# Use axis=0 if shape is (n_times, dim) or (n_times, n_particles, ...)
interp_func = interp1d(times, y_array, axis=0)

# Interpolate to regular times
y_regular = interp_func(t_regular)

These parameters show nucleation!

In [ ]:
plt.figure(figsize=(10,20))
sns.heatmap(np.array(y_regular)[:800,:], xticklabels=False, yticklabels=False, cbar=False)

Here's an animation

In [ ]:
from matplotlib.animation import PillowWriter
from animator import ParticleAnimator

interpolated_data = np.array(y_regular)[:800,:]
interpolated_data[-1] *= 0
reversed = 1-interpolated_data

#interpolated_data = np.vstack((interpolated_data,reversed))
total_time = len(interpolated_data)/fps

#plt.figure(figsize=(10,50))
#sns.heatmap(np.vstack((interpolated_data,reversed)), xticklabels=False, yticklabels=False, cbar=False)

fig, ax = plt.subplots()
plt.tight_layout()
fig.set_figwidth(10)
fig.set_figheight(2)
sns.heatmap([interpolated_data[0]], xticklabels=False, yticklabels=False, cbar=False, vmin=0, vmax=1, ax=ax)

plt.subplots_adjust(left=0, right=1, top=1, bottom=0)
plt.tight_layout(pad=0)


def update_ax(ax, data):
    ax.clear()
    i = data
    sns.heatmap([interpolated_data[i]], xticklabels=False, yticklabels=False, cbar=False, vmin=0, vmax=1, ax=ax)

filename = 'awesome_banner.gif'

anim = ParticleAnimator(fig, PillowWriter, filename)
anim.set_display(fps, total_time)
frame_data = anim.generate_frame_data(range(len(interpolated_data)))
anim.animate_ax(ax, update_ax, frame_data)
anim.make_animation()
anim.display()

In [ ]:
a = len(interpolated_data) 
b = 1000
b %= a
b